# 07 -- OOD/ambiguity eval and gate analysis

The direct test of the soft-mixing hypothesis: compares `hard_two_stage` (hard argmax routing) against the primary soft-gated MoE on dataset-ambiguous cross-dataset near-neighbor samples (and, if `evaluation.ood_holdout_dataset` is set, a genuinely held-out dataset). Also runs gate-weight/utilization diagnostics -- does the gate's learned partition line up with, diverge from, or refine ground-truth dataset identity?


## Setup

Run this cell first. It's the ONLY cell you should need to edit: change
`CONFIG_OVERRIDES` (a list of `--set key.path=value` style dotted overrides,
same syntax as `training.run`'s CLI) to narrow `data.active_datasets`,
switch `architecture`, point at a different Drive folder, etc.


In [ ]:
# ---- Single config cell: this is the only cell you should need to edit ----
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess, os
    REPO_DIR = '/content/dataset_moe_nids'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/selimsidan/dataset_moe_nids', REPO_DIR], check=True)
    %cd $REPO_DIR
    %pip install -q -r requirements.txt

CONFIG_PATH = 'config/default.yaml'

# Dotted --set overrides, same syntax as training.run's CLI. Narrow
# data.active_datasets to 2-3 datasets here for a fast iteration cycle --
# every downstream module (registry, harmonizer, expert-bank sizing,
# checkpoints) adapts automatically, no other code changes needed.
CONFIG_OVERRIDES = [
    # 'data.active_datasets=[NF-UNSW-NB15-v3,NF-BoT-IoT-v3]',
    # 'architecture=moe_dataset_soft',
    # 'training.device=cuda',
]

from training.config import load_config
config = load_config(CONFIG_PATH, CONFIG_OVERRIDES)
print('run_name:', config['run_name'])
print('architecture:', config['architecture'])
print('active_datasets:', config['data']['active_datasets'])
print('checkpoint_dir:', config['training']['checkpoint_dir'])


## OOD/ambiguity comparison

In [ ]:
from training.dataset import prepare_datasets
from training.stage_c_jointfinetune import build_model_from_checkpoints
from training.baseline_train import train_hard_two_stage
from evaluation.ood_ambiguity_eval import (
    build_cross_dataset_ambiguous_split, build_held_out_dataset_split, summarize_ood_comparison,
)
import torch

data = prepare_datasets(config)
device = torch.device(config['training'].get('device', 'cpu'))
moe_model = build_model_from_checkpoints(config, data, device)  # requires notebook 05
hard_two_stage_model = train_hard_two_stage(config, data)  # or reuse notebook 06's trained instance

split = build_cross_dataset_ambiguous_split(data, k_neighbors=5)
with torch.no_grad():
    preds_moe = moe_model.predict(torch.from_numpy(split.features)).numpy()
    preds_h2s = hard_two_stage_model.predict(torch.from_numpy(split.features)).numpy()

table = summarize_ood_comparison(split, {config['architecture']: preds_moe, 'hard_two_stage': preds_h2s})
print(table)

holdout = config['evaluation'].get('ood_holdout_dataset')
if holdout:
    ho_split = build_held_out_dataset_split(config, data, holdout)
    with torch.no_grad():
        preds_moe_ho = moe_model.predict(torch.from_numpy(ho_split.features)).numpy()
        preds_h2s_ho = hard_two_stage_model.predict(torch.from_numpy(ho_split.features)).numpy()
    print(summarize_ood_comparison(ho_split, {config['architecture']: preds_moe_ho, 'hard_two_stage': preds_h2s_ho}))
else:
    print('evaluation.ood_holdout_dataset not set -- skipping the genuinely-held-out-dataset leg.')


## Gate analysis: weight distribution per true dataset, utilization, collapse check

In [ ]:
from evaluation.gate_analysis import compute_gate_weights, gate_weight_by_true_dataset, detect_gate_collapse, expert_utilization

gate_weights = compute_gate_weights(moe_model, data.test.features)
by_true_dataset = gate_weight_by_true_dataset(gate_weights, data.active_datasets, data.test.dataset_name)
print(by_true_dataset)

print('\nexpert utilization:', expert_utilization(gate_weights, data.active_datasets))
collapse = detect_gate_collapse(gate_weights, data.active_datasets)
print('gate collapsed:', collapse.is_collapsed, '| collapsed experts:', collapse.collapsed_experts)
